# ST Model Analysis Notebook

Loads a trained spatio-temporal model (`GNNInductiveHeteroST`) and runs:
1. **Error analysis** – per-station spatial error maps, rankings, bias map, isolation plot
2. **Grid prediction** – interpolates rainfall onto a 1 km × 1 km grid using `predict_on_grid_st`
3. **Animation** – time-series heatmap animation of grid predictions

Key difference from `analysis.ipynb`: the ST model uses a context window of W preceding
timesteps (LSTM encoder), so the input data is rebuilt on a dense 15-min grid — exactly
as done during training in `train_st.py`.

In [ ]:
from src.sampling.main import stratified_spatial_kfold_dual  # must be first

import torch
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

from torch_geometric.loader import DataLoader as GeometricDataLoader

from models.gnn_st import GNNInductiveHeteroST
from src.utils import read_config
from src.raingauge.utils import load_raingauge_dataset
from src.radar.utils import load_processed_dataset
from src.cml.utils import load_cml_dataset
from src.graph.gaugegraphnew import GaugeGraphNew
from src.graph.radargraph import RadarGraph
from src.graph.cmlgraph import CMLGraph

from src.visualization.error_analysis import (
    analyze_station_errors,
    predict_on_test_stations_st,
    plot_station_rainfall_timeseries,
    get_viz_scales,
)
from src.visualization.grid_prediction import (
    predict_on_grid_st, plot_rainfall_grid, plot_rainfall_sequence, animate_rainfall_grid,
)
from src.visualization.main import visualise_singapore_outline, visualise_with_basemap

%load_ext autoreload
%autoreload 2

## Configuration

Set the experiment name and fold index here.

In [ ]:
# ── CHANGE THESE ──────────────────────────────────────────────────────────────
EXPERIMENT_NAME = "experiments/raingauge_cml_st1_window4_dropout10"
FOLD_IDX        = 0                      # which fold's weights to load
# ──────────────────────────────────────────────────────────────────────────────

EXPERIMENT_DIR = f"experiments/{EXPERIMENT_NAME}"
WEIGHTS_PATH   = f"{EXPERIMENT_DIR}/weather_gnn_best_{FOLD_IDX}.pth"

config       = read_config("config.yaml")
device       = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BOUNDS       = config["dataset_parameters"]["geography_bounds"]
FOLD_COUNT   = config["training_params"]["fold_count"]
DATA_SOURCES = config["datasources"]

tp          = config.get("temporal_params", {})
window_size = tp.get("window_size", 6)
lstm_hidden = tp.get("lstm_hidden", 32)
lstm_layers = tp.get("lstm_layers", 1)

# Analysis / time-window settings
_viz = config.get("analysis", {})
TIME_START       = _viz.get("time_start", None)
TIME_END         = _viz.get("time_end",   None)
SELECTED_STATION = _viz.get("selected_station", None)
TS_DISPLAY_START = _viz.get("ts_display_start", None)
TS_FRAME_COUNT   = _viz.get("ts_frame_count", None)

# Fixed visualisation scales — loaded from config.yaml visualisation section
SCALES          = get_viz_scales("config.yaml")
COMPARE_METRICS = SCALES["compare_metrics"]   # e.g. ['rmse', 'pearson_r']
METRIC_SCALES   = SCALES["metric_scales"]     # {metric: {vmin, vmax, boundaries}}
RAINFALL_SCALE  = SCALES["rainfall"]          # {vmin, vmax, boundaries}

print(f"Experiment  : {EXPERIMENT_NAME}")
print(f"Weights     : {WEIGHTS_PATH}")
print(f"Device      : {device}")
print(f"Bounds      : {BOUNDS}")
print(f"Folds       : {FOLD_COUNT}")
print(f"Data sources: {DATA_SOURCES}")
print(f"window_size={window_size}  lstm_hidden={lstm_hidden}  lstm_layers={lstm_layers}")
print(f"Time window : {TIME_START} → {TIME_END}  (None = full period)")
print(f"Compare     : {COMPARE_METRICS}")
print(f"Rainfall scale: vmin={RAINFALL_SCALE['vmin']}, vmax={RAINFALL_SCALE['vmax']}")

## Load Datasets (Dense 15-min Grid)

The ST model was trained on a complete 15-min grid (left-join rather than inner-join).
Rebuilding the same grid ensures graph topology and feature shapes match the checkpoint.

In [ ]:
uptime_threshold = config["filters"]["uptime_threshold"]
start_year       = config["dataset_parameters"]["start_year"]
end_year         = config["dataset_parameters"]["end_year"]

# ── Raingauge: left-join onto dense 15-min grid ───────────────────────────────
raingauge_df_raw, mapping_df = load_raingauge_dataset(
    start=start_year, end=end_year, uptime_threshold=uptime_threshold
)
raingauge_df_raw = raingauge_df_raw.reset_index().sort_values("timestamp").reset_index(drop=True)

all_ts            = pd.to_datetime(raingauge_df_raw["timestamp"])
complete_ts_index = pd.date_range(start=all_ts.min(), end=all_ts.max(), freq="15min")
complete_df       = pd.DataFrame({"timestamp": complete_ts_index})
print(f"Dense grid: {len(complete_ts_index)} timesteps  ({complete_ts_index[0]} → {complete_ts_index[-1]})")

raingauge_deduped = raingauge_df_raw.drop_duplicates("timestamp")
raingauge_df      = complete_df.merge(raingauge_deduped, on="timestamp", how="left").reset_index(drop=True)

# ── Radar: left-join + per-timestep validity flag ────────────────────────────
if "radar" in DATA_SOURCES:
    radar_df_raw = load_processed_dataset("database/processed_radar_dataset.pkl")
    radar_df_raw = radar_df_raw.sort_values("timestamp").reset_index(drop=True)
    radar_deduped  = radar_df_raw.drop_duplicates("timestamp")
    valid_radar_ts = set(pd.to_datetime(radar_deduped["timestamp"]))
    radar_valid_flags = np.array(
        [1.0 if ts in valid_radar_ts else 0.0 for ts in complete_ts_index], dtype=np.float32
    )
    first_valid  = radar_deduped["data"].dropna().iloc[0]
    zero_arr     = np.zeros_like(first_valid)
    radar_full   = complete_df.merge(radar_deduped, on="timestamp", how="left")
    radar_full["data"]   = [v if isinstance(v, np.ndarray) else zero_arr.copy() for v in radar_full["data"]]
    radar_full["bounds"] = radar_full["bounds"].ffill().bfill()
    radar_df = radar_full
    print(f"Radar: {int(radar_valid_flags.sum())} / {len(complete_ts_index)} timesteps with real data")
else:
    radar_df, radar_valid_flags = None, None

# ── CML: expand to complete (timestamp × link × station) grid ────────────────
def _expand_cml_to_dense_grid(cml_df, complete_df):
    _STATIC = ["link_id", "station", "site_a_latitude", "site_a_longitude",
               "site_b_latitude", "site_b_longitude", "length", "frequency", "polarization"]
    _INDEX  = ["timestamp", "link_id", "station"]
    present_static  = [c for c in _STATIC if c in cml_df.columns]
    cml_static      = cml_df[present_static].drop_duplicates(["link_id", "station"]).reset_index(drop=True)
    _static_non_idx = [c for c in present_static if c not in ["link_id", "station"]]
    dynamic_cols    = [c for c in cml_df.columns if c not in _static_non_idx]
    cml_dynamic     = cml_df[dynamic_cols].drop_duplicates(_INDEX)
    expanded = complete_df.merge(cml_static[["link_id", "station"]], how="cross")
    expanded = expanded.merge(cml_dynamic, on=_INDEX, how="left")
    expanded = expanded.merge(cml_static, on=["link_id", "station"], how="left", suffixes=("_drop", ""))
    for col in _static_non_idx:
        if f"{col}_drop" in expanded.columns:
            expanded = expanded.drop(columns=[f"{col}_drop"])
    _check = [c for c in cml_dynamic.columns if c not in _INDEX]
    if _check:
        expanded["valid"] = expanded[_check[0]].notna().astype(float)
    else:
        expanded["valid"] = 1.0
    return expanded.fillna(0)

if "cml" in DATA_SOURCES:
    cml_file = config["dataset_parameters"]["cml_folder"]
    cml_df_raw, cml_coordinates_df = load_cml_dataset(cml_file)
    cml_df_raw = cml_df_raw.sort_values("timestamp").reset_index(drop=True)
    cml_df = _expand_cml_to_dense_grid(cml_df_raw, complete_df)
    print(f"CML: {len(cml_df)} rows  ({int(cml_df['valid'].sum())} real, {int((cml_df['valid']==0).sum())} padded)")
else:
    cml_df, cml_coordinates_df = None, None

print(f"Raingauge : {raingauge_df.shape}")
print(f"Mapping df: {mapping_df.shape}")

## Build Graph for Each Fold

In [ ]:
layer_cfg  = config["layer_connect"]

split_info = stratified_spatial_kfold_dual(
    mapping_df, seed=config["training_params"]["seed"], plot=False, n_splits=FOLD_COUNT
)

gauge_graph_arr = []
for i in range(FOLD_COUNT):
    gauge_graph = GaugeGraphNew(
        raingauge_df, mapping_df,
        split_info=split_info[i],
        knn=layer_cfg["gauge_gauge"],
    )

    if "radar" in DATA_SOURCES and radar_df is not None:
        radar_graph      = RadarGraph(radar_df)
        radar_heterodata = radar_graph.get_radar_heterodata()
        # Append per-timestep validity flag — matches train_st.py setup
        N_r     = radar_heterodata["radar"].x.shape[0]
        T_r     = radar_heterodata["radar"].x.shape[1]
        valid_t = torch.tensor(radar_valid_flags).view(1, T_r, 1).expand(N_r, -1, -1)
        radar_heterodata["radar"].x = torch.cat([radar_heterodata["radar"].x, valid_t], dim=-1)
        gauge_graph.add_heterodata(
            heterodata_layer=radar_heterodata,
            coords=radar_graph.grid_coords,
            layer_name="radar",
            knn=layer_cfg["radar_gauge"],
        )

    if "cml" in DATA_SOURCES and cml_df is not None:
        cml_graph      = CMLGraph(cml_df, cml_coordinates_df)
        cml_heterodata = cml_graph.get_heterodata()
        gauge_graph.add_heterodata(
            heterodata_layer=cml_heterodata,
            coords=cml_coordinates_df,
            layer_name="cml",
            knn=layer_cfg["cml_gauge"],
        )

    gauge_graph_arr.append(gauge_graph)

print("Graphs built for all folds.")

## Infer Model Architecture from Checkpoint

Reads `hidden_channels`, `num_layers`, `lstm_hidden`, `lstm_layers`, and `in_channels_dict`
from the saved state-dict so the model definition matches the checkpoint exactly.

In [ ]:
def infer_arch_st(weights_path: str):
    """Return (num_layers, hidden_channels, lstm_hidden, lstm_layers, in_channels_dict)"""
    sd = torch.load(weights_path, map_location="cpu")

    conv_keys       = [k for k in sd if k.startswith("convs.")]
    num_layers      = max(int(k.split(".")[1]) for k in conv_keys) + 1
    hidden_channels = sd["lin.weight"].shape[1]
    lstm_h          = sd["lstm_encoders.raingauge.weight_hh_l0"].shape[1]
    lstm_l          = sum(1 for k in sd if k.startswith("lstm_encoders.raingauge.weight_hh_l"))

    in_ch = {}
    for k, v in sd.items():
        if "lstm_encoders." in k and k.endswith(".weight_ih_l0"):
            ntype = k.split(".")[1]
            in_ch[ntype] = v.shape[1]

    return num_layers, hidden_channels, lstm_h, lstm_l, in_ch

_DATA_FEATURE_DIM = 2
num_layers, hidden_channels, lstm_hidden_ck, lstm_layers_ck, in_channels_dict = infer_arch_st(WEIGHTS_PATH)

raingauge_in = in_channels_dict.get("raingauge", _DATA_FEATURE_DIM)
lpe_k        = raingauge_in - _DATA_FEATURE_DIM

print(f"num_layers      = {num_layers}")
print(f"hidden_channels = {hidden_channels}")
print(f"lstm_hidden     = {lstm_hidden_ck}  (config: {lstm_hidden})")
print(f"lstm_layers     = {lstm_layers_ck}  (config: {lstm_layers})")
print(f"in_channels     = {in_channels_dict}")
print(f"lpe_k           = {lpe_k}")

## Instantiate & Load Model

In [ ]:
def compute_norm_stats(heterodata):
    stats = {}
    for node_type in heterodata.node_types:
        x    = heterodata[node_type].x   # [N, T, F]
        mean = x.mean(dim=(0, 1))
        std  = x.std(dim=(0, 1)).clamp(min=1e-8)
        stats[node_type] = (mean, std)
    return stats


def apply_norm(heterodata, stats):
    """Apply pre-computed normalisation stats. Only touches .x, never .y."""
    normed = heterodata.clone()
    for node_type in heterodata.node_types:
        if node_type in stats:
            mean, std = stats[node_type]
            normed[node_type].x = (heterodata[node_type].x - mean) / std
    return normed


test_heterodata = gauge_graph_arr[FOLD_IDX].get_test_heterodata()
edge_types      = test_heterodata.edge_types

model = GNNInductiveHeteroST(
    in_channels_dict=in_channels_dict,
    hidden_channels=hidden_channels,
    out_channels=1,
    num_layers=num_layers,
    edge_types=edge_types,
    window_size=window_size,
    lstm_hidden=lstm_hidden_ck,
    lstm_layers=lstm_layers_ck,
).to(device)

model.load_state_dict(torch.load(WEIGHTS_PATH, map_location=device))
model.eval()
print(f"Model loaded from {WEIGHTS_PATH}")
print(model)

---
# Part 1 – Error Analysis

Uses the pre-computed `per_station_metrics_f*.csv` files saved during testing.

In [ ]:
import glob as _glob

csv_paths = sorted(_glob.glob(f"{EXPERIMENT_DIR}/per_station_metrics_f*.csv"))
print("Found metric CSVs:")
for p in csv_paths:
    print(" ", p)

In [ ]:
train_ids = split_info[FOLD_IDX]["ml"]["train"]

station_df = analyze_station_errors(
    csv_paths=csv_paths,
    mapping_df=mapping_df,
)
station_df.head()

---
## Per-Station Rainfall Time Series

The ST model's test predictions are collected per station from the normalised
`test_heterodata_n` produced in Part 2 (Grid Prediction).  Run Part 2 first to
populate `pred_timestamps` and `station_predictions_df`.

Use `TIME_START` / `TIME_END` (set in the config cell or `config.yaml`) to zoom
into a specific event window.

In [ ]:
# Requires test_heterodata_n and pred_timestamps from Part 2 (Grid Prediction).
# Run that section first if not already done.
print("Running per-station ST inference …")
station_predictions_df, station_actuals_df = predict_on_test_stations_st(
    model=model,
    heterodata=test_heterodata_n,
    mapping_df=mapping_df,
    timestamps=pd.Series(complete_ts_index),
    window_size=window_size,
    device=device,
)
print(f"Done.  Shape: {station_predictions_df.shape}  (timesteps × test stations)")
print(f"Test stations: {list(station_predictions_df.columns)}")

In [ ]:
# ── Choose which station to plot ─────────────────────────────────────────────
_first_test = list(station_predictions_df.columns)[0]
STATION_ID  = SELECTED_STATION if SELECTED_STATION in station_predictions_df.columns \
              else _first_test

print(f"Plotting station : {STATION_ID}")
print(f"Time window      : {TIME_START or 'start'} → {TIME_END or 'end'}")

fig, ax = plt.subplots(figsize=(16, 4))
plot_station_rainfall_timeseries(
    predictions_df=station_predictions_df,
    actuals_df=station_actuals_df,
    station_id=STATION_ID,
    time_start=TIME_START,
    time_end=TIME_END,
    title=f"{EXPERIMENT_NAME} (ST) — Station {STATION_ID}",
    ax=ax,
)
plt.tight_layout()
plt.show()

### Individual plots (inline)

In [ ]:
from src.visualization.error_analysis import (
    plot_spatial_error_map,
    plot_error_ranking,
    plot_bias_map,
    plot_error_vs_isolation,
)

m1, m2 = COMPARE_METRICS
fig, axes = plt.subplots(1, 2, figsize=(18, 7))
plot_spatial_error_map(station_df, BOUNDS, metric=m1, ax=axes[0],
                       title=f"Per-station {m1.upper()}",
                       **METRIC_SCALES.get(m1, {}))
plot_spatial_error_map(station_df, BOUNDS, metric=m2, ax=axes[1],
                       title=f"Per-station {m2.upper()}",
                       **METRIC_SCALES.get(m2, {}))
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))
plot_bias_map(station_df, BOUNDS, top_n_labels=10, ax=ax)
plt.tight_layout()
plt.show()

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
plot_error_ranking(station_df, metric="mae", top_n=15, ax=ax1)
plot_error_ranking(station_df, metric="f1",  top_n=15, ax=ax2)
fig.suptitle("Worst-performing stations — ST model")
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
plot_error_vs_isolation(station_df, train_ids, metric="mae", ax=ax)
plt.tight_layout()
plt.show()

### Spatial Error Maps — with Basemap + Singapore Outline

In [ ]:
m1, m2 = COMPARE_METRICS
fig, axes = plt.subplots(1, 2, figsize=(18, 7))
plot_spatial_error_map(station_df, BOUNDS, metric=m1, ax=axes[0],
                       title=f"Per-station {m1.upper()} (with basemap)",
                       **METRIC_SCALES.get(m1, {}))
plot_spatial_error_map(station_df, BOUNDS, metric=m2, ax=axes[1],
                       title=f"Per-station {m2.upper()} (with basemap)",
                       **METRIC_SCALES.get(m2, {}))
for ax in axes:
    visualise_with_basemap(ax=ax)
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))
plot_bias_map(station_df, BOUNDS, top_n_labels=10, ax=ax)
visualise_singapore_outline(ax=ax)
visualise_with_basemap(ax=ax)
plt.tight_layout()
plt.show()

---
# Part 2 – Grid Prediction

Runs the ST model over every valid timestep in the test graph (starting from
`window_size`) and interpolates predictions onto a regular 1 km grid.
`predictions[i]` corresponds to `pred_timestamps.iloc[i]`.

In [ ]:
# ── Grid prediction settings ──────────────────────────────────────────────────
RESOLUTION_KM = 1.0   # grid spacing in km
KNN_GAUGE     = 5     # how many real gauges each grid point connects to
INCLUDE_LPE   = lpe_k > 0
print(f"include_lpe={INCLUDE_LPE}  lpe_k={lpe_k}  window_size={window_size}")

In [ ]:
# Normalise test data using training statistics (matches train_st.py)
train_heterodata  = gauge_graph_arr[FOLD_IDX].get_train_heterodata()
norm_stats        = compute_norm_stats(train_heterodata)
test_heterodata_n = apply_norm(gauge_graph_arr[FOLD_IDX].get_test_heterodata(), norm_stats)

# Timestamps aligned to predictions (predictions[i] ↔ pred_timestamps.iloc[i])
pred_timestamps = pd.Series(complete_ts_index[window_size:])

print(f"Running ST grid prediction on fold {FOLD_IDX} …")
predictions, grid_coords, grid_shape = predict_on_grid_st(
    model=model,
    heterodata=test_heterodata_n,
    mapping_df=mapping_df,
    bounds=BOUNDS,
    window_size=window_size,
    resolution_km=RESOLUTION_KM,
    knn_gauge=KNN_GAUGE,
    device=str(device),
    include_lpe=INCLUDE_LPE,
    lpe_k=lpe_k,
)

T_pred = predictions.shape[0]
print(f"Grid shape    : {grid_shape}  ({grid_shape[0]} rows × {grid_shape[1]} cols)")
print(f"Pred timesteps: {T_pred}  (valid range: [0, {T_pred}))")
print(f"Pred range    : [{predictions.min():.3f}, {predictions.max():.3f}] mm")

In [ ]:
# ── Single timestep ──────────────────────────────────────────────────────────
TIMESTEP = 100   # index into predictions [0, T_pred)

fig, ax = plt.subplots(figsize=(10, 8))
plot_rainfall_grid(
    predictions, grid_shape, BOUNDS,
    timestamp_idx=TIMESTEP,
    mapping_df=mapping_df,
    title=f"Predicted Rainfall — {pred_timestamps.iloc[TIMESTEP]}",
    ax=ax,
    **RAINFALL_SCALE,
)
plt.tight_layout()
plt.show()

In [ ]:
# ── Sequence of timesteps ─────────────────────────────────────────────────────
SEQ_START = 100
timestep_indices = list(range(SEQ_START, SEQ_START + 6))
seq_titles = [str(pred_timestamps.iloc[t]) for t in timestep_indices]

fig = plot_rainfall_sequence(
    predictions, grid_shape, BOUNDS,
    timestep_indices=timestep_indices,
    mapping_df=mapping_df,
    titles=seq_titles,
    save_path=f"{EXPERIMENT_DIR}/analysis/grid_sequence_st.png",
    **RAINFALL_SCALE,
)
plt.show()

In [ ]:
# ── Mean rainfall over entire test period ─────────────────────────────────────
mean_pred    = predictions.mean(axis=0)          # [n_rows, n_cols]
mean_pred_4d = mean_pred[np.newaxis, :, :]        # [1, n_rows, n_cols]

fig, ax = plt.subplots(figsize=(10, 8))
plot_rainfall_grid(
    mean_pred_4d, grid_shape, BOUNDS,
    timestamp_idx=0,
    mapping_df=mapping_df,
    title="Mean predicted rainfall over test period (ST model)",
    ax=ax,
    **RAINFALL_SCALE,
)
plt.tight_layout()
plt.show()

## Time Series Animation

Animate the predicted rainfall grid as a spatial heatmap over time.
`animate_rainfall_grid` returns a `FuncAnimation`; render it inline with `HTML(anim.to_jshtml())`.

Note: `ANIM_START` is an index into `predictions` (0-based).
The corresponding real timestamp is `pred_timestamps.iloc[ANIM_START]`.

In [ ]:
print(BOUNDS)

In [ ]:
from IPython.display import HTML

# ── Animation settings ────────────────────────────────────────────────────────
N_FRAMES    = 120
INTERVAL_MS = 150
ANIM_START  = 100   # index into predictions [0, T_pred)

anim_indices    = list(range(ANIM_START, ANIM_START + N_FRAMES))
anim_timestamps = pred_timestamps.iloc[anim_indices].tolist()

import os; os.makedirs(f"{EXPERIMENT_DIR}/analysis", exist_ok=True)

anim = animate_rainfall_grid(
    predictions, grid_shape, BOUNDS,
    timestep_indices=anim_indices,
    timestamps=anim_timestamps,
    mapping_df=mapping_df,
    interval_ms=INTERVAL_MS,
    save_path=f"{EXPERIMENT_DIR}/analysis/rainfall_animation_st.gif",
    **RAINFALL_SCALE,
)

HTML(anim.to_jshtml())